# EuroSAT — dataset measurements

Records the facts that `docs/eurosat-pipeline.md` leaves marked **verify**, so they
live in the repository rather than in a terminal scrollback.

The logic lives in `src/` (protocol §2). This notebook measures and draws; it does
not define anything the training code imports.

## 0. Environment

Stamped every run. The repo venv is Python 3.13 — if `sys.executable` below is not
`.venv\Scripts\python.exe`, the VS Code kernel selector is pointed at a different
interpreter and nothing recorded here is trustworthy.

In [ ]:
import sys
import matplotlib
import torch
import torchvision

print("python     ", sys.version.split()[0])
print("executable ", sys.executable)
print("torch      ", torch.__version__)
print("torchvision", torchvision.__version__)
print("matplotlib ", matplotlib.__version__)
print("cuda       ", torch.cuda.is_available(),
      torch.cuda.get_device_capability() if torch.cuda.is_available() else None)

KeyboardInterrupt: 

: 

## 1. Download provenance

Protocol §3 flagged the download host as unconfirmed for this torchvision version.
This settles it for whatever is actually installed.

In [ ]:
import inspect
from torchvision.datasets import EuroSAT

print(inspect.getsource(EuroSAT.download))

## 2. Size and class distribution

Protocol §3: *"Per-class counts are not uniform and remain unmeasured … One `Counter`
over `ds.samples`, recorded in the notebook. It decides whether the split must be
stratified and whether plain accuracy is an adequate metric."*

In [ ]:
from collections import Counter
from pathlib import Path

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

ds = EuroSAT(root=str(REPO / "data"))

counts = Counter(label for _, label in ds.samples)
by_class = {name: counts[i] for i, name in enumerate(ds.classes)}

print("N        ", len(ds))
print("classes  ", len(ds.classes))
for name, n in by_class.items():
    print(f"  {name:<22} {n:>5}")
print("min/max  ", min(by_class.values()), max(by_class.values()),
      f"(ratio {max(by_class.values()) / min(by_class.values()):.2f})")

**Measured 18 September 2026.** N = 27,000 across ten classes, in clean multiples of
500 — itself evidence the extract completed, since a partial extraction gives ragged
counts. Imbalance is 1.5:1 (Pasture 2,000 against 3,000 for five classes).

**Decision — the split is stratified.** Not because 1.5:1 is dangerous, but because
of the two-seed design in §4. Test is 4,050 images; Pasture's share under an
unstratified split is hypergeometric with mean 300 and sd ≈ 15. Harmless for accuracy,
but it would make `split_seed` change both *which* images land in test and *how many
per class*. Stratifying makes that knob mean one thing. Always on; not a config field,
because a field would imply it is an axis to sweep.

**Decision — plain accuracy is adequate** at this imbalance. Per-class accuracy is
logged anyway: it costs nothing and a single class near 0% is the signature of a
broken label mapping, which is the failure P2 exists to catch.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.0, 3.4))
names = list(by_class)
ax.bar(range(len(names)), [by_class[n] for n in names], color="#4C72B0", width=0.68)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_ylabel("images")
ax.set_title(f"EuroSAT class counts (N = {len(ds):,})")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3, linewidth=0.6)
ax.set_axisbelow(True)
fig.tight_layout()

out = REPO / "figures" / "eurosat-class-counts.png"
out.parent.mkdir(exist_ok=True)
fig.savefig(out, dpi=150, metadata={"Software": None})
print("wrote", out)

`metadata={"Software": None}` strips the matplotlib version from the PNG `tEXt` chunk.
Without it, figures drawn by different matplotlib versions differ byte-wise with
bit-identical pixels, and the repo shows a diff for a figure nothing changed about.

## 3. File ordering — why P2 depends on it

`ImageFolder.make_dataset` builds its file list with `for fname in sorted(fnames)`:
a plain lexicographic string sort. EuroSAT filenames carry an unpadded integer, so
lexicographic and numeric order disagree from the second element onwards.

`EuroSATRaw` must reproduce the lexicographic order. Sorting numerically — the more
obviously *correct* thing to do — makes index *i* a different image in each
implementation, and P2 fails on the sampled pairs while both datasets are
individually fine.

In [ ]:
folder = REPO / "data" / "eurosat" / "2750" / "AnnualCrop"
names = [p.name for p in folder.iterdir() if p.suffix == ".jpg"]

lexicographic = sorted(names)
numeric = sorted(names, key=lambda s: int(s.rsplit("_", 1)[1].split(".")[0]))

print("torchvision order (sorted):      ", lexicographic[:6])
print("numeric order (what NOT to use): ", numeric[:6])
print("first index where they differ:   ",
      next(i for i, (a, b) in enumerate(zip(lexicographic, numeric)) if a != b))